# Validação e Sanity Checks - Camada Intermediate (BA)

## Pergunta de Pesquisa
**Nas escolas da Bahia (BA) em 2025, escolas com maior nível de complexidade de gestão apresentam maior proporção de docentes em altos níveis de esforço (níveis 5+6)?**

Este notebook valida os modelos intermediate antes de montar a camada serving.

### Checklist
- [ ] Filtro BA aplicado corretamente (select distinct uf)
- [ ] Contagens: escolas na staging vs intermediate
- [ ] Percentuais de null em colunas-chave
- [ ] Joins não perderam escolas
- [ ] Hipótese exploratória: complexidade vs esforço docente
- [ ] Testes dbt passaram

# Setup e Conexão

In [1]:
import duckdb
import pandas as pd
import numpy as np

DB_PATH = 'project_data.duckdb'
con = duckdb.connect(database=DB_PATH, read_only=True)

# 1. Validar Filtro BA

In [2]:
# Verificar que apenas BA está presente
query_uf = """select distinct uf from main_intermediate.int_escolas_indicadores_ba"""
df_uf = con.execute(query_uf).fetchdf()
print("UFs presentes em int_escolas_indicadores_ba:")
print(df_uf)

if len(df_uf) == 1 and df_uf['uf'][0] == 'BA':
    print("✓ Filtro BA aplicado corretamente")
else:
    print("✗ Erro: dados de outras UFs encontrados!")

UFs presentes em int_escolas_indicadores_ba:
   uf
0  BA
✓ Filtro BA aplicado corretamente


# 2. Contagens: Staging vs Intermediate

In [3]:
# Comparar contagens entre staging (fonte) e models intermediate
query_count = """
-- ICG em BA
select 'stg_icg_escolas (BA)' as source, count(distinct id_escola) as qtd
from main_staging.stg_icg_escolas
where uf='BA'

union all

-- int_escolas_indicadores_ba
select 'int_escolas_indicadores_ba' as source, count(distinct id_escola) as qtd
from main_intermediate.int_escolas_indicadores_ba

union all

-- int_perfil_gestao_ba
select 'int_perfil_gestao_ba' as source, count(distinct id_escola) as qtd
from main_intermediate.int_perfil_gestao_ba
"""

df_count = con.execute(query_count).fetchdf()
print("\nContagens por fonte:")
print(df_count.to_string(index=False))

# Análise
count_icg = df_count[df_count['source'] == 'stg_icg_escolas (BA)']['qtd'].values[0]
count_int_indic = df_count[df_count['source'] == 'int_escolas_indicadores_ba']['qtd'].values[0]
count_int_gestao = df_count[df_count['source'] == 'int_perfil_gestao_ba']['qtd'].values[0]

if count_icg == count_int_indic:
    print(f"✓ int_escolas_indicadores_ba manteve todas as {count_icg} escolas do ICG")
else:
    print(f"⚠ int_escolas_indicadores_ba: {count_icg} (ICG) -> {count_int_indic} (intermediate)")

print(f"✓ int_perfil_gestao_ba tem {count_int_gestao} escolas (com gestores agregados)")


Contagens por fonte:
                    source   qtd
      stg_icg_escolas (BA) 15854
int_escolas_indicadores_ba 15854
      int_perfil_gestao_ba 15854
✓ int_escolas_indicadores_ba manteve todas as 15854 escolas do ICG
✓ int_perfil_gestao_ba tem 15854 escolas (com gestores agregados)


# 3. Percentual de Nulls em Colunas-Chave

In [4]:
# Percentual de nulls
query_nulls = """
select
    round(100.0 * count(*) filter (where ano_censo is null) / count(*), 2) as pct_null_ano_censo,
    round(100.0 * count(*) filter (where id_escola is null) / count(*), 2) as pct_null_id_escola,
    round(100.0 * count(*) filter (where nivel_complexidade_gestao is null) / count(*), 2) as pct_null_complexidade,
    round(100.0 * count(*) filter (where pct_docentes_alto_esforco is null) / count(*), 2) as pct_null_alto_esforco
from main_intermediate.int_escolas_indicadores_ba
"""

df_nulls = con.execute(query_nulls).fetchdf()
print("\nPercentual de nulls em int_escolas_indicadores_ba:")
print(df_nulls.to_string(index=False))

if df_nulls['pct_null_alto_esforco'].values[0] > 0:
    print(f"\n⚠ {df_nulls['pct_null_alto_esforco'].values[0]}% das escolas não têm dados IED (esforço docente)")
    print("  Isso é esperado se nem todas as escolas ICG têm correspondência em IED.")


Percentual de nulls em int_escolas_indicadores_ba:
 pct_null_ano_censo  pct_null_id_escola  pct_null_complexidade  pct_null_alto_esforco
                0.0                 0.0                    0.0                    0.0


# 4. Exploração Rápida: Hipótese Preliminar

In [5]:
# Teste rápido da hipótese: complexidade vs esforço docente
query_hyp = """
select
    nivel_complexidade_gestao,
    count(distinct id_escola) as qtd_escolas,
    round(avg(pct_docentes_alto_esforco), 2) as media_pct_alto_esforco,
    round(min(pct_docentes_alto_esforco), 2) as min_pct,
    round(max(pct_docentes_alto_esforco), 2) as max_pct
from main_intermediate.int_escolas_indicadores_ba
where pct_docentes_alto_esforco is not null
group by nivel_complexidade_gestao
order by nivel_complexidade_gestao
"""

df_hyp = con.execute(query_hyp).fetchdf()
print("\nHipótese Exploratória: Complexidade de Gestão vs Esforço Docente (Alto)")
print(df_hyp.to_string(index=False))
print("\n💡 Observação: Comportamento esperado é tendência de aumento na média conforme nível de complexidade sobe.")


Hipótese Exploratória: Complexidade de Gestão vs Esforço Docente (Alto)
 nivel_complexidade_gestao  qtd_escolas  media_pct_alto_esforco  min_pct  max_pct
                         1         3994                    0.69      0.0    100.0
                         2         4395                    1.78      0.0    100.0
                         3         2119                    5.27      0.0    100.0
                         4         2433                    3.36      0.0    100.0
                         5         2608                    5.70      0.0    100.0
                         6          305                    9.11      0.0     66.6

💡 Observação: Comportamento esperado é tendência de aumento na média conforme nível de complexidade sobe.


# 5. Amostra de Dados

In [6]:
# Amostra de int_escolas_indicadores_ba
query_sample = """select * from main_intermediate.int_escolas_indicadores_ba limit 10"""
df_sample = con.execute(query_sample).fetchdf()
print("\nAmostra: int_escolas_indicadores_ba")
print(df_sample.to_string())


Amostra: int_escolas_indicadores_ba
   ano_censo  id_escola  uf                                                                         nome_escola  id_municipio nome_municipio  nivel_complexidade_gestao  pct_docentes_esforco_nivel_1  pct_docentes_esforco_nivel_2  pct_docentes_esforco_nivel_3  pct_docentes_esforco_nivel_4  pct_docentes_esforco_nivel_5  pct_docentes_esforco_nivel_6  pct_docentes_alto_esforco
0       2025   29211590  BA                        COLEGIO ESTADUAL DOUTOR FRANCISCO ROCHA FILHO TEMPO INTEGRAL       2900108         Abaíra                          4                           NaN                           NaN                           NaN                           NaN                           NaN                           NaN                        0.0
1       2025   29211956  BA                                                      GRUPO ESCOLAR HORACIO DE MATOS       2900108         Abaíra                          2                          83.3                

In [7]:
# Amostra de int_perfil_gestao_ba
query_sample_gestao = """select * from main_intermediate.int_perfil_gestao_ba limit 10"""
df_sample_gestao = con.execute(query_sample_gestao).fetchdf()
print("\nAmostra: int_perfil_gestao_ba")
print(df_sample_gestao.to_string())


Amostra: int_perfil_gestao_ba
   ano_censo  id_escola  uf                                nome_escola  qtd_gestores_total  qtd_gestores_feminino  qtd_gestores_masculino  qtd_gestores_brancos  qtd_gestores_pretos  qtd_gestores_pardos  qtd_gestores_graduacao  qtd_gestores_pos_especializacao  qtd_gestores_pos_mestrado  qtd_gestores_pos_doutorado  prop_pos_especializacao
0       2025   29001790  BA                     ESCOLA JARDIM IMPERIAL                 1.0                    1.0                     0.0                   1.0                  0.0                  0.0                     1.0                              1.0                        0.0                         0.0                      1.0
1       2025   29002028  BA        ESCOLA MUNICIPAL OTTOMAR SCHWENGBER                 1.0                    1.0                     0.0                   1.0                  0.0                  0.0                     1.0                              1.0                        0.0      

### Comandos úteis para executar no terminal para confirmar testes:
```bash
dbt test --select intermediate
dbt docs generate && dbt docs serve
```

In [8]:
con.close()
print("\n✓ Validação concluída. Conexão fechada.")


✓ Validação concluída. Conexão fechada.
